<a href="https://colab.research.google.com/github/jayatigupta05/VEP/blob/main/GenoHybrid_VEP_2_Foundation_and_Sequence_Encoding.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GenoHybrid-VEP — Notebook 2
## Tri-Modal Model Training

This notebook loads the cleaned feature table produced by Notebook 1 and begins the final GenoHybrid-VEP model pipeline.

## Imports, reproducibility and device

In [1]:
import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import (
    GroupShuffleSplit,
    GroupKFold
)

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    matthews_corrcoef,
    classification_report,
    confusion_matrix,
    roc_auc_score
)

from sklearn.preprocessing import StandardScaler

from tqdm.auto import tqdm


# ============================================================
# REPRODUCIBILITY
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


# ============================================================
# DEVICE
# ============================================================

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

Device: cpu


## Mount Google Drive and load Notebook 1 output

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_PATH = Path(
    "/content/drive/MyDrive/"
    "variant_effect_prediction"
)

DATA_PATH = (
    PROJECT_PATH /
    "data" /
    "processed" /
    "genohybrid_features.parquet"
)

CHECKPOINT_DIR = (
    PROJECT_PATH /
    "checkpoints"
)

RESULTS_DIR = (
    PROJECT_PATH /
    "results"
)

CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


df = pd.read_parquet(
    DATA_PATH
)


print(
    "Dataset shape:",
    df.shape
)

print(
    "\nColumns:"
)

print(
    df.columns.tolist()
)

Dataset shape: (100000, 36)

Columns:
['node_id', 'variant_key', 'Chromosome', 'Start', 'Stop', 'ReferenceAlleleVCF', 'AlternateAlleleVCF', 'GeneSymbol', 'Type', 'ClinicalSignificance', 'Label', 'Assembly', 'OriginSimple', 'ReviewStatus', 'NumberSubmitters', 'SomaticClinicalImpact', 'Ref_A', 'Ref_C', 'Ref_G', 'Ref_T', 'Alt_A', 'Alt_C', 'Alt_G', 'Alt_T', 'Tab_Scaled_Submitters', 'Tab_Origin_Numeric', 'Tab_Review_Score', 'Tab_Somatic_Impact', 'seq201', 'phylop', 'phastcons', 'molecular_consequence_raw', 'consequence_category', 'gnomad_af', 'gnomad_present', 'ref_match']


## Dataset audit

This cell checks that the output from Notebook 1 is valid before any modeling starts.

Expected:
- `seq201` should always have length 201
- every `ref_match` should be `True`
- `node_id` should be unique
- labels should only be 0, 1, 2
- gnomAD may still be completely missing at this stage

In [4]:
print(
    "================ DATASET AUDIT ================"
)

print(
    "\nVariants:",
    f"{len(df):,}"
)


print(
    "\nClass counts:"
)

print(
    df["Label"]
    .value_counts()
    .sort_index()
)


print(
    "\nClass percentages:"
)

print(
    (
        df["Label"]
        .value_counts(
            normalize=True
        )
        .sort_index()
        * 100
    ).round(2)
)


print(
    "\nSequence lengths:"
)

print(
    df["seq201"]
    .str.len()
    .value_counts()
    .head()
)


print(
    "\nREF match:"
)

print(
    df["ref_match"]
    .value_counts(
        dropna=False
    )
)


print(
    "\nMissing biological features:"
)

for column in [
    "phylop",
    "phastcons",
    "consequence_category",
    "gnomad_af"
]:

    if column in df.columns:
        missing = (
            df[column]
            .isna()
            .mean()
        )

        print(
            f"{column:25s}: "
            f"{missing:.2%}"
        )

    else:
        print(
            f"{column:25s}: "
            "COLUMN MISSING"
        )


# ============================================================
# HARD CHECKS
# ============================================================

assert (
    df["seq201"]
    .str.len()
    .eq(201)
    .all()
), "Some seq201 rows do not have length 201."

assert (
    df["ref_match"]
    .all()
), "Some REF alleles do not match GRCh38."

assert (
    df["node_id"]
    .is_unique
), "node_id is not unique."

assert (
    set(
        df["Label"]
        .unique()
    )
    <= {0, 1, 2}
), "Unexpected labels found."


print(
    "\n✅ Dataset passed core checks."
)

================ DATASET AUDIT ================

Variants: 100,000

Class counts:
Label
0    34621
1     5135
2    60244
Name: count, dtype: int64

Class percentages:
Label
0    34.62
1     5.14
2    60.24
Name: proportion, dtype: float64

Sequence lengths:
seq201
201    100000
Name: count, dtype: int64

REF match:
ref_match
True    100000
Name: count, dtype: int64

Missing biological features:
phylop                   : 0.00%
phastcons                : 0.00%
consequence_category     : 0.00%
gnomad_af                : 100.00%

✅ Dataset passed core checks.


## Build the gene-grouping variable

The final project evaluates on genes unseen during training.

For variants without a usable gene symbol, we create a fallback group based on:

`NO_GENE_<chromosome>_<megabase region>`

This is safer than treating every gene-less variant independently.

In [6]:
df["group_id"] = (
    df["GeneSymbol"]
    .astype("string")
)


missing_gene = (
    df["group_id"].isna()
    |
    (
        df["group_id"]
        .fillna("")
        .str.len()
        == 0
    )
)


df.loc[
    missing_gene,
    "group_id"
] = (
    "NO_GENE_"
    +
    df.loc[
        missing_gene,
        "Chromosome"
    ].astype(str)
    +
    "_"
    +
    (
        df.loc[
            missing_gene,
            "Start"
        ]
        // 1_000_000
    ).astype(str)
)


print(
    "Unique groups:",
    df["group_id"]
    .nunique()
)

print(
    "Variants without named genes:",
    int(missing_gene.sum())
)

Unique groups: 14465
Variants without named genes: 0


## Development train / validation / test split

For development, we use a single gene-separated split:

- about 80% training genes
- about 10% validation genes
- about 10% test genes

Later, after the full architecture is stable, this will be replaced with the planned 5-fold `GroupKFold`.

In [7]:
# ============================================================
# OUTER SPLIT
# train+validation vs test
# ============================================================

outer_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.10,
    random_state=SEED
)


trainval_idx, test_idx = next(

    outer_split.split(
        df,
        y=df["Label"],
        groups=df["group_id"]
    )
)


trainval_df = (
    df.iloc[
        trainval_idx
    ]
    .copy()
)


test_df = (
    df.iloc[
        test_idx
    ]
    .copy()
)


# ============================================================
# INNER SPLIT
# train vs validation
# ============================================================

inner_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.1111,
    random_state=SEED
)


train_idx, val_idx = next(

    inner_split.split(
        trainval_df,
        y=trainval_df["Label"],
        groups=trainval_df["group_id"]
    )
)


train_df = (
    trainval_df.iloc[
        train_idx
    ]
    .copy()
)


val_df = (
    trainval_df.iloc[
        val_idx
    ]
    .copy()
)


print(
    "Train:",
    len(train_df)
)

print(
    "Validation:",
    len(val_df)
)

print(
    "Test:",
    len(test_df)
)

Train: 80065
Validation: 10298
Test: 9637


## Prove there is no gene-group leakage

In [8]:
train_groups = set(
    train_df["group_id"]
)

val_groups = set(
    val_df["group_id"]
)

test_groups = set(
    test_df["group_id"]
)


train_val_overlap = (
    train_groups
    &
    val_groups
)

train_test_overlap = (
    train_groups
    &
    test_groups
)

val_test_overlap = (
    val_groups
    &
    test_groups
)


print(
    "Train ↔ Validation overlap:",
    len(train_val_overlap)
)

print(
    "Train ↔ Test overlap:",
    len(train_test_overlap)
)

print(
    "Validation ↔ Test overlap:",
    len(val_test_overlap)
)


assert (
    len(train_val_overlap)
    == 0
)

assert (
    len(train_test_overlap)
    == 0
)

assert (
    len(val_test_overlap)
    == 0
)


print(
    "\n✅ No gene-group leakage."
)

Train ↔ Validation overlap: 0
Train ↔ Test overlap: 0
Validation ↔ Test overlap: 0

✅ No gene-group leakage.


## Cell 8 — Inspect class balance in each partition

In [9]:
def describe_split(
    name,
    dataframe
):

    print(
        f"\n========== {name} =========="
    )

    print(
        dataframe["Label"]
        .value_counts()
        .sort_index()
    )

    print(
        (
            dataframe["Label"]
            .value_counts(
                normalize=True
            )
            .sort_index()
            * 100
        ).round(2)
    )


describe_split(
    "TRAIN",
    train_df
)

describe_split(
    "VALIDATION",
    val_df
)

describe_split(
    "TEST",
    test_df
)


========== TRAIN ==========
Label
0    27686
1     4110
2    48269
Name: count, dtype: int64
Label
0    34.58
1     5.13
2    60.29
Name: proportion, dtype: float64

========== VALIDATION ==========
Label
0    3500
1     549
2    6249
Name: count, dtype: int64
Label
0    33.99
1     5.33
2    60.68
Name: proportion, dtype: float64

========== TEST ==========
Label
0    3435
1     476
2    5726
Name: count, dtype: int64
Label
0    35.64
1     4.94
2    59.42
Name: proportion, dtype: float64


# Arm 1 Preparation — 101 bp DNA sequence input

Notebook 1 stores a 201 bp GRCh38 sequence centered on each variant.

For the final sequence arm we crop the central 101 bp:

- 50 bases before the variant
- the variant position
- 50 bases after the variant

The surrounding sequence is ordinary one-hot DNA.

At the centre position we use a mutation-aware encoding:

- `+1` on the reference base channel
- `-1` on the alternative base channel


## Cell 9 — Mutation-aware sequence encoder

In [10]:
BASE_TO_INDEX = {
    "A": 0,
    "C": 1,
    "G": 2,
    "T": 3
}


def encode_variant_sequence(
    seq201,
    ref,
    alt,
    window_size=101
):

    assert (
        window_size % 2
        == 1
    )

    assert (
        ref in BASE_TO_INDEX
        and
        alt in BASE_TO_INDEX
    ), "REF and ALT must be A/C/G/T."

    center_201 = 100

    half = (
        window_size
        // 2
    )


    sequence = seq201[
        center_201 - half:
        center_201 + half + 1
    ]


    assert (
        len(sequence)
        == window_size
    )


    encoded = np.zeros(
        (
            4,
            window_size
        ),
        dtype=np.float32
    )


    # Ordinary one-hot DNA context
    for position, base in enumerate(
        sequence
    ):

        if base in BASE_TO_INDEX:

            encoded[
                BASE_TO_INDEX[base],
                position
            ] = 1.0


    # ========================================================
    # Mutation-aware centre encoding
    # ========================================================

    centre = half

    encoded[
        :,
        centre
    ] = 0.0


    encoded[
        BASE_TO_INDEX[ref],
        centre
    ] = 1.0


    encoded[
        BASE_TO_INDEX[alt],
        centre
    ] = -1.0


    return encoded

## Cell 10 — Test the sequence encoding

This is the checkpoint for Part 1.

Expected:
- shape `(4, 101)`
- one positive centre channel for REF
- one negative centre channel for ALT

In [11]:
sample = train_df.iloc[0]


encoded = encode_variant_sequence(

    sample["seq201"],

    sample[
        "ReferenceAlleleVCF"
    ],

    sample[
        "AlternateAlleleVCF"
    ],

    window_size=101
)


print(
    "Encoded shape:",
    encoded.shape
)

print(
    "REF:",
    sample[
        "ReferenceAlleleVCF"
    ]
)

print(
    "ALT:",
    sample[
        "AlternateAlleleVCF"
    ]
)

print(
    "Centre encoding:",
    encoded[:, 50]
)


assert encoded.shape == (4, 101)
assert np.isclose(
    encoded[
        BASE_TO_INDEX[
            sample["ReferenceAlleleVCF"]
        ],
        50
    ],
    1.0
)
assert np.isclose(
    encoded[
        BASE_TO_INDEX[
            sample["AlternateAlleleVCF"]
        ],
        50
    ],
    -1.0
)

print(
    "\n✅ 101-bp mutation-aware sequence encoding works."
)

Encoded shape: (4, 101)
REF: G
ALT: A
Centre encoding: [-1.  0.  1.  0.]

✅ 101-bp mutation-aware sequence encoding works.


## Dataset for sequence arm

In [12]:
class SequenceDataset(Dataset):

    def __init__(self, dataframe):

        self.df = dataframe.reset_index(drop=True)

        self.labels = torch.tensor(
            self.df["Label"].values,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.df)


    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        seq = encode_variant_sequence(
            row["seq201"],
            row["ReferenceAlleleVCF"],
            row["AlternateAlleleVCF"],
            window_size=101
        )
        seq = torch.tensor(seq, dtype=torch.float32)
        label = self.labels[idx]

        return seq, label

## Create train, validation and test datasets

In [13]:
train_dataset = SequenceDataset(
    train_df
)

val_dataset = SequenceDataset(
    val_df
)

test_dataset = SequenceDataset(
    test_df
)


print(
    "Train samples:",
    len(train_dataset)
)

print(
    "Validation samples:",
    len(val_dataset)
)

print(
    "Test samples:",
    len(test_dataset)
)

Train samples: 80065
Validation samples: 10298
Test samples: 9637


## DataLoaders

In [14]:
if torch.cuda.is_available():

    BATCH_SIZE = 512

else:

    BATCH_SIZE = 256


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)


print(
    "Batch size:",
    BATCH_SIZE
)

print(
    "Train batches:",
    len(train_loader)
)

print(
    "Validation batches:",
    len(val_loader)
)

print(
    "Test batches:",
    len(test_loader)
)

Batch size: 256
Train batches: 313
Validation batches: 41
Test batches: 38


## Calculate class weights

In [16]:
train_counts = (train_df["Label"].value_counts().sort_index())
num_classes = 3

num_train = len(train_df)
class_weights = []

for class_id in range(num_classes):
    count = train_counts.get(class_id, 1)
    weight = (num_train / (num_classes * count))
    class_weights.append(weight)

class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

print("Class weights:", class_weights)

Class weights: tensor([0.9640, 6.4935, 0.5529])


## 101 bp CNN arm

In [17]:
class SequenceCNN(nn.Module):

    def __init__(
        self,
        embedding_dim=64
    ):

        super().__init__()


        # ==========================================
        # Convolution Block 1
        # ==========================================

        self.conv1 = nn.Conv1d(
            in_channels=4,
            out_channels=32,
            kernel_size=7,
            padding=3
        )

        self.bn1 = nn.BatchNorm1d(
            32
        )


        # ==========================================
        # Convolution Block 2
        # ==========================================

        self.conv2 = nn.Conv1d(
            in_channels=32,
            out_channels=64,
            kernel_size=5,
            padding=2
        )

        self.bn2 = nn.BatchNorm1d(
            64
        )


        # ==========================================
        # Convolution Block 3
        # ==========================================

        self.conv3 = nn.Conv1d(
            in_channels=64,
            out_channels=64,
            kernel_size=3,
            padding=1
        )

        self.bn3 = nn.BatchNorm1d(
            64
        )


        self.pool = nn.MaxPool1d(
            kernel_size=2
        )


        self.dropout = nn.Dropout(
            0.30
        )


        # Global pooling produces:
        #
        # average = 64
        # maximum = 64
        #
        # total = 128

        self.embedding = nn.Linear(
            128,
            embedding_dim
        )


    def forward(
        self,
        x
    ):

        # Input:
        #
        # [batch, 4, 101]


        x = self.conv1(x)

        x = self.bn1(x)

        x = F.relu(x)

        x = self.pool(x)



        x = self.conv2(x)

        x = self.bn2(x)

        x = F.relu(x)

        x = self.pool(x)



        x = self.conv3(x)

        x = self.bn3(x)

        x = F.relu(x)



        # ==========================================
        # Global pooling
        # ==========================================

        avg_pool = torch.mean(
            x,
            dim=2
        )

        max_pool = torch.max(
            x,
            dim=2
        ).values


        x = torch.cat(
            [
                avg_pool,
                max_pool
            ],
            dim=1
        )


        x = self.dropout(x)


        embedding = self.embedding(
            x
        )

        embedding = F.relu(
            embedding
        )


        return embedding

## Sequence only classifier

In [18]:
class SequenceOnlyModel(nn.Module):

    def __init__(
        self,
        num_classes=3
    ):

        super().__init__()


        self.encoder = SequenceCNN(
            embedding_dim=64
        )


        self.classifier = nn.Sequential(

            nn.Linear(
                64,
                32
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        embedding = self.encoder(
            x
        )


        logits = self.classifier(
            embedding
        )


        return logits

## Model initialisation

In [19]:
sequence_model = SequenceOnlyModel(
    num_classes=3
).to(device)


loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)


optimizer = torch.optim.AdamW(

    sequence_model.parameters(),

    lr=1e-3,

    weight_decay=1e-4
)


scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(

    optimizer,

    mode="max",

    factor=0.5,

    patience=1
)


print(
    sequence_model
)


total_parameters = sum(

    p.numel()

    for p in sequence_model.parameters()

    if p.requires_grad

)


print(
    "\nTrainable parameters:",
    f"{total_parameters:,}"
)

SequenceOnlyModel(
  (encoder): SequenceCNN(
    (conv1): Conv1d(4, 32, kernel_size=(7,), stride=(1,), padding=(3,))
    (bn1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv2): Conv1d(32, 64, kernel_size=(5,), stride=(1,), padding=(2,))
    (bn2): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (conv3): Conv1d(64, 64, kernel_size=(3,), stride=(1,), padding=(1,))
    (bn3): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (pool): MaxPool1d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (dropout): Dropout(p=0.3, inplace=False)
    (embedding): Linear(in_features=128, out_features=64, bias=True)
  )
  (classifier): Sequential(
    (0): Linear(in_features=64, out_features=32, bias=True)
    (1): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.3, inplace=False)
    (4): Linear(in_features

## Initial validation

In [20]:
seq_batch, labels = next(
    iter(train_loader)
)


seq_batch = seq_batch.to(
    device
)


labels = labels.to(
    device
)


with torch.no_grad():

    logits = sequence_model(
        seq_batch
    )


print(
    "Input shape:",
    seq_batch.shape
)

print(
    "Output shape:",
    logits.shape
)

print(
    "Label shape:",
    labels.shape
)


assert (
    seq_batch.shape[1:]
    ==
    (4, 101)
)


assert (
    logits.shape[1]
    ==
    3
)


print(
    "\n✅ CNN forward pass works."
)

Input shape: torch.Size([256, 4, 101])
Output shape: torch.Size([256, 3])
Label shape: torch.Size([256])

✅ CNN forward pass works.


## Evaluation loop

In [21]:
@torch.no_grad()
def evaluate_model(
    model,
    loader
):

    model.eval()


    all_targets = []

    all_predictions = []

    all_probabilities = []


    for seq, targets in loader:

        seq = seq.to(
            device
        )

        targets = targets.to(
            device
        )


        logits = model(
            seq
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        all_targets.extend(
            targets.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


    y_true = np.asarray(
        all_targets
    )

    y_pred = np.asarray(
        all_predictions
    )

    y_prob = np.asarray(
        all_probabilities
    )


    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )


    accuracy = accuracy_score(
        y_true,
        y_pred
    )


    balanced_accuracy = balanced_accuracy_score(
        y_true,
        y_pred
    )


    mcc = matthews_corrcoef(
        y_true,
        y_pred
    )


    return {

        "accuracy":
            accuracy,

        "balanced_accuracy":
            balanced_accuracy,

        "macro_f1":
            macro_f1,

        "mcc":
            mcc,

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }

## Training loop

In [22]:
EPOCHS = 15

PATIENCE = 3


best_val_f1 = -1

epochs_without_improvement = 0


best_model_path = (
    CHECKPOINT_DIR /
    "sequence_cnn_best.pt"
)


print(
    f"Training sequence CNN on {device}..."
)


for epoch in range(
    EPOCHS
):


    # ==========================================
    # TRAIN
    # ==========================================

    sequence_model.train()


    running_loss = 0.0


    progress = tqdm(
        train_loader,
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )


    for seq, targets in progress:


        seq = seq.to(
            device,
            non_blocking=True
        )


        targets = targets.to(
            device,
            non_blocking=True
        )


        optimizer.zero_grad(
            set_to_none=True
        )


        logits = sequence_model(
            seq
        )


        loss = loss_fn(
            logits,
            targets
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            sequence_model.parameters(),

            max_norm=5.0

        )


        optimizer.step()


        running_loss += (
            loss.item()
            *
            seq.size(0)
        )


        progress.set_postfix(
            loss=f"{loss.item():.4f}"
        )


    train_loss = (
        running_loss
        /
        len(train_dataset)
    )


    # ==========================================
    # VALIDATE
    # ==========================================

    val_results = evaluate_model(

        sequence_model,

        val_loader

    )


    val_f1 = (
        val_results[
            "macro_f1"
        ]
    )


    scheduler.step(
        val_f1
    )


    current_lr = (
        optimizer
        .param_groups[0]["lr"]
    )


    print(

        f"Epoch {epoch + 1:02d} | "
        f"Loss: {train_loss:.4f} | "
        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "
        f"Val Balanced Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "
        f"Val Macro-F1: "
        f"{val_f1:.4f} | "
        f"Val MCC: "
        f"{val_results['mcc']:.4f} | "
        f"LR: {current_lr:.2e}"

    )


    # ==========================================
    # EARLY STOPPING
    # ==========================================

    if (
        val_f1
        >
        best_val_f1
        + 1e-4
    ):


        best_val_f1 = val_f1


        epochs_without_improvement = 0


        torch.save(

            {
                "model_state_dict":
                    sequence_model.state_dict(),

                "val_macro_f1":
                    best_val_f1,

                "epoch":
                    epoch + 1,

                "seed":
                    SEED,

                "window_size":
                    101

            },

            best_model_path

        )


        print(
            "✓ Best model saved."
        )


    else:


        epochs_without_improvement += 1


        print(

            "No improvement:",

            f"{epochs_without_improvement}/"
            f"{PATIENCE}"

        )


    if (
        epochs_without_improvement
        >=
        PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

Training sequence CNN on cpu...


Epoch 1/15:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 01 | Loss: 1.1102 | Val Acc: 0.2481 | Val Balanced Acc: 0.3547 | Val Macro-F1: 0.2380 | Val MCC: 0.0266 | LR: 1.00e-03
✓ Best model saved.


Epoch 2/15:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 02 | Loss: 1.0936 | Val Acc: 0.2770 | Val Balanced Acc: 0.3595 | Val Macro-F1: 0.2559 | Val MCC: 0.0414 | LR: 1.00e-03
✓ Best model saved.


...

Epoch 8/15:   0%|          | 0/313 [00:00<?, ?it/s]

Exception ignored in: Exception ignored in: <function _MultiProcessingDataLoaderIter.__del__ at 0x7d2969e46b60><function _MultiProcessingDataLoaderIter.__del__ at 0x7d2969e46b60>

Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
Exception ignored in: Traceback (most recent call last):
      File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
<function _MultiProcessingDataLoaderIter.__del__ at 0x7d2969e46b60>self._shutdown_workers()    

self._shutdown_workers()  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers
Traceback (most recent call last):

  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1709, in __del__
        if w.is_alive():  File "/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py", line 1692, in _shutdown_workers

self._s

Epoch 08 | Loss: 0.9691 | Val Acc: 0.3536 | Val Balanced Acc: 0.4929 | Val Macro-F1: 0.3326 | Val MCC: 0.1248 | LR: 5.00e-04
No improvement: 2/3


Epoch 9/15:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 09 | Loss: 0.9365 | Val Acc: 0.4290 | Val Balanced Acc: 0.5067 | Val Macro-F1: 0.3837 | Val MCC: 0.1499 | LR: 5.00e-04
No improvement: 3/3

Early stopping triggered.


## Evaluating against previous best model

In [26]:
checkpoint = torch.load(
    best_model_path,
    map_location=device
)


sequence_model.load_state_dict(
    checkpoint[
        "model_state_dict"
    ]
)


print(
    "Restored epoch:",
    checkpoint["epoch"]
)

train_results = evaluate_model(
    sequence_model,
    train_loader
)

print(
    "Train Macro-F1:",
    train_results["macro_f1"]
)

print(
    "Validation Macro-F1:",
    checkpoint["val_macro_f1"]
)

Restored epoch: 6
Train Macro-F1: 0.3952936615532914
Validation Macro-F1: 0.3875698245256758


# Arm 2: MLP

## Arm 2 Features

In [27]:
# ============================================================
# ARM 2 — BIOMARKER FEATURES
# ============================================================

NUMERIC_FEATURES = [
    "phylop",
    "phastcons",
    "Tab_Review_Score",
    "Tab_Origin_Numeric",
    "Tab_Somatic_Impact"
]

CATEGORICAL_FEATURES = [
    "consequence_category"
]


print("Numeric features:")
print(NUMERIC_FEATURES)

print("\nCategorical features:")
print(CATEGORICAL_FEATURES)

Numeric features:
['phylop', 'phastcons', 'Tab_Review_Score', 'Tab_Origin_Numeric', 'Tab_Somatic_Impact']

Categorical features:
['consequence_category']


In [29]:
# ============================================================
# FIT NUMERIC IMPUTATION ON TRAIN ONLY
# ============================================================

train_medians = {}


for feature in NUMERIC_FEATURES:

    median = (
        train_df[feature]
        .median()
    )

    # If an entire feature somehow happens to be missing,
    # use zero rather than crashing.
    if pd.isna(median):
        median = 0.0

    train_medians[
        feature
    ] = float(median)


print(
    "Training medians:"
)

print(
    train_medians
)

Training medians:
{'phylop': 1.4950000047683716, 'phastcons': 0.9909999966621399, 'Tab_Review_Score': 1.0, 'Tab_Origin_Numeric': 1.0, 'Tab_Somatic_Impact': 1.0}


In [30]:
train_consequence_categories = sorted(

    train_df[
        "consequence_category"
    ]
    .fillna("unknown")
    .astype(str)
    .unique()
)


print(
    "Training consequence categories:"
)

print(
    train_consequence_categories
)

Training consequence categories:
['intron', 'missense', 'nonsense_or_stop', 'other', 'splice', 'synonymous', 'unknown', 'utr']


## Biomarker transformation function

In [31]:
def build_biomarker_matrix(
    dataframe,
    fit_scaler=False,
    scaler=None
):

    dataframe = (
        dataframe
        .reset_index(drop=True)
        .copy()
    )


    # ========================================================
    # NUMERIC FEATURES
    # ========================================================

    numeric = dataframe[
        NUMERIC_FEATURES
    ].copy()


    # Add missing-value indicators before imputation.
    #
    # Missing conservation itself may carry useful information.
    for feature in [
        "phylop",
        "phastcons"
    ]:

        numeric[
            f"{feature}_missing"
        ] = (
            dataframe[feature]
            .isna()
            .astype(np.float32)
        )


    # Fill missing values using medians learned from TRAIN only.
    for feature in NUMERIC_FEATURES:

        numeric[feature] = (
            numeric[feature]
            .fillna(
                train_medians[feature]
            )
        )


    numeric = numeric.astype(
        np.float32
    )


    # ========================================================
    # SCALE NUMERIC FEATURES
    # ========================================================

    if fit_scaler:

        scaler = StandardScaler()

        numeric_scaled = scaler.fit_transform(
            numeric
        )

    else:

        numeric_scaled = scaler.transform(
            numeric
        )


    # ========================================================
    # CATEGORICAL: MOLECULAR CONSEQUENCE
    # ========================================================

    consequence = (
        dataframe[
            "consequence_category"
        ]
        .fillna("unknown")
        .astype(str)
    )


    consequence_features = np.zeros(

        (
            len(dataframe),
            len(
                train_consequence_categories
            )
        ),

        dtype=np.float32
    )


    category_to_index = {

        category: i

        for i, category
        in enumerate(
            train_consequence_categories
        )
    }


    for row_index, value in enumerate(
        consequence
    ):

        if value in category_to_index:

            consequence_features[
                row_index,
                category_to_index[value]
            ] = 1.0


    # ========================================================
    # FINAL MATRIX
    # ========================================================

    biomarker_matrix = np.concatenate(

        [
            numeric_scaled.astype(
                np.float32
            ),

            consequence_features
        ],

        axis=1
    )


    return (
        biomarker_matrix,
        scaler
    )

## Build train / validation / test biomarker matrices

In [32]:
X_train_tab, tab_scaler = (
    build_biomarker_matrix(
        train_df,
        fit_scaler=True
    )
)


X_val_tab, _ = (
    build_biomarker_matrix(
        val_df,
        fit_scaler=False,
        scaler=tab_scaler
    )
)


X_test_tab, _ = (
    build_biomarker_matrix(
        test_df,
        fit_scaler=False,
        scaler=tab_scaler
    )
)


print(
    "Train:",
    X_train_tab.shape
)

print(
    "Validation:",
    X_val_tab.shape
)

print(
    "Test:",
    X_test_tab.shape
)


assert (
    X_train_tab.shape[1]
    ==
    X_val_tab.shape[1]
    ==
    X_test_tab.shape[1]
)


assert np.isfinite(
    X_train_tab
).all()


assert np.isfinite(
    X_val_tab
).all()


assert np.isfinite(
    X_test_tab
).all()


print(
    "\n✅ Biomarker matrices ready."
)

Train: (80065, 15)
Validation: (10298, 15)
Test: (9637, 15)

✅ Biomarker matrices ready.


## Create biomarker datasets

In [33]:
class BiomarkerDataset(
    Dataset
):

    def __init__(
        self,
        X,
        labels
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )


        self.y = torch.tensor(
            labels,
            dtype=torch.long
        )


    def __len__(
        self
    ):

        return len(
            self.y
        )


    def __getitem__(
        self,
        idx
    ):

        return (
            self.X[idx],
            self.y[idx]
        )

In [34]:
train_tab_dataset = BiomarkerDataset(

    X_train_tab,

    train_df[
        "Label"
    ].values
)


val_tab_dataset = BiomarkerDataset(

    X_val_tab,

    val_df[
        "Label"
    ].values
)


test_tab_dataset = BiomarkerDataset(

    X_test_tab,

    test_df[
        "Label"
    ].values
)


print(
    "Train:",
    len(
        train_tab_dataset
    )
)

print(
    "Validation:",
    len(
        val_tab_dataset
    )
)

print(
    "Test:",
    len(
        test_tab_dataset
    )
)

Train: 80065
Validation: 10298
Test: 9637


## Biomarker dataloader

In [35]:
TAB_BATCH_SIZE = 512


train_tab_loader = DataLoader(

    train_tab_dataset,

    batch_size=TAB_BATCH_SIZE,

    shuffle=True,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


val_tab_loader = DataLoader(

    val_tab_dataset,

    batch_size=TAB_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


test_tab_loader = DataLoader(

    test_tab_dataset,

    batch_size=TAB_BATCH_SIZE,

    shuffle=False,

    num_workers=0,

    pin_memory=torch.cuda.is_available()
)


print(
    "Biomarker input dimension:",
    X_train_tab.shape[1]
)

print(
    "Train batches:",
    len(
        train_tab_loader
    )
)

Biomarker input dimension: 15
Train batches: 157


## Arm 2 MLP

In [36]:
class BiomarkerMLP(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        embedding_dim=32
    ):

        super().__init__()


        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                64
            ),

            nn.BatchNorm1d(
                64
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),


            nn.Linear(
                64,
                32
            ),

            nn.BatchNorm1d(
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.20
            ),


            nn.Linear(
                32,
                embedding_dim
            ),

            nn.ReLU()
        )


    def forward(
        self,
        x
    ):

        return self.network(
            x
        )

## Biomarker only classifier

In [37]:
class BiomarkerOnlyModel(
    nn.Module
):

    def __init__(
        self,
        input_dim,
        num_classes=3
    ):

        super().__init__()


        self.encoder = BiomarkerMLP(

            input_dim=input_dim,

            embedding_dim=32
        )


        self.classifier = nn.Sequential(

            nn.Linear(
                32,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        x
    ):

        embedding = self.encoder(
            x
        )


        logits = self.classifier(
            embedding
        )


        return logits

## Initialise and validate Arm 2

In [38]:
biomarker_model = BiomarkerOnlyModel(

    input_dim=
        X_train_tab.shape[1],

    num_classes=3

).to(device)


tab_loss_fn = (
    nn.CrossEntropyLoss(
        weight=class_weights
    )
)


tab_optimizer = (
    torch.optim.AdamW(

        biomarker_model.parameters(),

        lr=1e-3,

        weight_decay=1e-4
    )
)


tab_scheduler = (
    torch.optim.lr_scheduler
    .ReduceLROnPlateau(

        tab_optimizer,

        mode="max",

        factor=0.5,

        patience=1
    )
)


print(
    biomarker_model
)


trainable = sum(

    p.numel()

    for p in biomarker_model.parameters()

    if p.requires_grad
)


print(
    "\nTrainable parameters:",
    f"{trainable:,}"
)

BiomarkerOnlyModel(
  (encoder): BiomarkerMLP(
    (network): Sequential(
      (0): Linear(in_features=15, out_features=64, bias=True)
      (1): BatchNorm1d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU()
      (3): Dropout(p=0.3, inplace=False)
      (4): Linear(in_features=64, out_features=32, bias=True)
      (5): BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (6): ReLU()
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=32, out_features=32, bias=True)
      (9): ReLU()
    )
  )
  (classifier): Sequential(
    (0): Linear(in_features=32, out_features=32, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.3, inplace=False)
    (3): Linear(in_features=32, out_features=3, bias=True)
  )
)

Trainable parameters: 5,507


In [39]:
tab_batch, labels = next(

    iter(
        train_tab_loader
    )
)


tab_batch = tab_batch.to(
    device
)


labels = labels.to(
    device
)


with torch.no_grad():

    logits = biomarker_model(
        tab_batch
    )


print(
    "Input:",
    tab_batch.shape
)

print(
    "Output:",
    logits.shape
)

print(
    "Labels:",
    labels.shape
)


assert (
    logits.shape[1]
    == 3
)


print(
    "\n✅ Biomarker MLP forward pass works."
)

Input: torch.Size([512, 15])
Output: torch.Size([512, 3])
Labels: torch.Size([512])

✅ Biomarker MLP forward pass works.


## Evaluation function

In [40]:
@torch.no_grad()
def evaluate_tabular_model(
    model,
    loader
):

    model.eval()


    all_targets = []
    all_predictions = []
    all_probabilities = []


    for features, targets in loader:

        features = features.to(
            device
        )

        targets = targets.to(
            device
        )


        logits = model(
            features
        )


        probabilities = torch.softmax(
            logits,
            dim=1
        )


        predictions = torch.argmax(
            probabilities,
            dim=1
        )


        all_targets.extend(
            targets.cpu().numpy()
        )

        all_predictions.extend(
            predictions.cpu().numpy()
        )

        all_probabilities.extend(
            probabilities.cpu().numpy()
        )


    y_true = np.asarray(
        all_targets
    )

    y_pred = np.asarray(
        all_predictions
    )

    y_prob = np.asarray(
        all_probabilities
    )


    return {

        "accuracy":
            accuracy_score(
                y_true,
                y_pred
            ),

        "balanced_accuracy":
            balanced_accuracy_score(
                y_true,
                y_pred
            ),

        "macro_f1":
            f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0
            ),

        "mcc":
            matthews_corrcoef(
                y_true,
                y_pred
            ),

        "y_true":
            y_true,

        "y_pred":
            y_pred,

        "y_prob":
            y_prob
    }

## Training

In [41]:
TAB_EPOCHS = 15

TAB_PATIENCE = 3


best_tab_f1 = -1.0

tab_epochs_without_improvement = 0


best_tab_path = (

    CHECKPOINT_DIR /

    "biomarker_mlp_best.pt"
)


print(
    f"Training biomarker MLP on {device}..."
)


for epoch in range(
    TAB_EPOCHS
):

    biomarker_model.train()


    running_loss = 0.0


    progress = tqdm(

        train_tab_loader,

        desc=
            f"Epoch "
            f"{epoch + 1}/"
            f"{TAB_EPOCHS}"
    )


    for features, targets in progress:


        features = features.to(
            device,
            non_blocking=True
        )


        targets = targets.to(
            device,
            non_blocking=True
        )


        tab_optimizer.zero_grad(
            set_to_none=True
        )


        logits = biomarker_model(
            features
        )


        loss = tab_loss_fn(
            logits,
            targets
        )


        loss.backward()


        torch.nn.utils.clip_grad_norm_(

            biomarker_model.parameters(),

            5.0
        )


        tab_optimizer.step()


        running_loss += (

            loss.item()

            *
            features.size(0)
        )


        progress.set_postfix(

            loss=
                f"{loss.item():.4f}"
        )


    train_loss = (

        running_loss

        /

        len(
            train_tab_dataset
        )
    )


    val_results = (
        evaluate_tabular_model(

            biomarker_model,

            val_tab_loader
        )
    )


    val_f1 = (
        val_results[
            "macro_f1"
        ]
    )


    tab_scheduler.step(
        val_f1
    )


    current_lr = (
        tab_optimizer
        .param_groups[0]["lr"]
    )


    print(

        f"Epoch {epoch + 1:02d} | "

        f"Loss: "
        f"{train_loss:.4f} | "

        f"Val Acc: "
        f"{val_results['accuracy']:.4f} | "

        f"Val Balanced Acc: "
        f"{val_results['balanced_accuracy']:.4f} | "

        f"Val Macro-F1: "
        f"{val_f1:.4f} | "

        f"Val MCC: "
        f"{val_results['mcc']:.4f} | "

        f"LR: "
        f"{current_lr:.2e}"
    )


    if (
        val_f1
        >
        best_tab_f1
        + 1e-4
    ):

        best_tab_f1 = (
            val_f1
        )

        tab_epochs_without_improvement = 0


        torch.save(

            {
                "model_state_dict":
                    biomarker_model
                    .state_dict(),

                "val_macro_f1":
                    best_tab_f1,

                "epoch":
                    epoch + 1,

                "input_dim":
                    X_train_tab.shape[1],

                "numeric_features":
                    NUMERIC_FEATURES,

                "consequence_categories":
                    train_consequence_categories,

                "seed":
                    SEED
            },

            best_tab_path
        )


        print(
            "✓ Best model saved."
        )


    else:

        tab_epochs_without_improvement += 1


        print(

            "No improvement:",

            f"{tab_epochs_without_improvement}/"
            f"{TAB_PATIENCE}"
        )


    if (
        tab_epochs_without_improvement
        >=
        TAB_PATIENCE
    ):

        print(
            "\nEarly stopping triggered."
        )

        break

Training biomarker MLP on cpu...


Epoch 1/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.6723 | Val Acc: 0.8781 | Val Balanced Acc: 0.8275 | Val Macro-F1: 0.7857 | Val MCC: 0.7691 | LR: 1.00e-03
✓ Best model saved.


Epoch 2/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 02 | Loss: 0.4752 | Val Acc: 0.8797 | Val Balanced Acc: 0.8305 | Val Macro-F1: 0.7897 | Val MCC: 0.7717 | LR: 1.00e-03
✓ Best model saved.


...

Epoch 06 | Loss: 0.4576 | Val Acc: 0.8776 | Val Balanced Acc: 0.8303 | Val Macro-F1: 0.7856 | Val MCC: 0.7688 | LR: 1.00e-03
No improvement: 1/3


Epoch 7/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 07 | Loss: 0.4558 | Val Acc: 0.8775 | Val Balanced Acc: 0.8307 | Val Macro-F1: 0.7847 | Val MCC: 0.7687 | LR: 5.00e-04
No improvement: 2/3


Epoch 8/15:   0%|          | 0/157 [00:00<?, ?it/s]

Epoch 08 | Loss: 0.4558 | Val Acc: 0.8791 | Val Balanced Acc: 0.8303 | Val Macro-F1: 0.7869 | Val MCC: 0.7712 | LR: 5.00e-04
No improvement: 3/3

Early stopping triggered.


## Restoring best model

In [42]:
tab_checkpoint = torch.load(

    best_tab_path,

    map_location=device
)


biomarker_model.load_state_dict(

    tab_checkpoint[
        "model_state_dict"
    ]
)


print(
    "Best epoch:",
    tab_checkpoint[
        "epoch"
    ]
)

print(
    "Best validation Macro-F1:",
    tab_checkpoint[
        "val_macro_f1"
    ]
)

Best epoch: 5
Best validation Macro-F1: 0.7935789316048032


## Train / validation / test comparison

In [43]:
train_tab_results = (
    evaluate_tabular_model(

        biomarker_model,

        train_tab_loader
    )
)


val_tab_results = (
    evaluate_tabular_model(

        biomarker_model,

        val_tab_loader
    )
)


test_tab_results = (
    evaluate_tabular_model(

        biomarker_model,

        test_tab_loader
    )
)


print(
    "=========== BIOMARKER MLP ONLY ==========="
)


print(
    f"Train Macro-F1: "
    f"{train_tab_results['macro_f1']:.4f}"
)


print(
    f"Validation Macro-F1: "
    f"{val_tab_results['macro_f1']:.4f}"
)


print(
    f"Test Macro-F1: "
    f"{test_tab_results['macro_f1']:.4f}"
)


print(
    f"Test Accuracy: "
    f"{test_tab_results['accuracy'] * 100:.2f}%"
)


print(
    f"Test Balanced Accuracy: "
    f"{test_tab_results['balanced_accuracy']:.4f}"
)


print(
    f"Test MCC: "
    f"{test_tab_results['mcc']:.4f}"
)

=========== BIOMARKER MLP ONLY ===========
Train Macro-F1: 0.7937
Validation Macro-F1: 0.7936
Test Macro-F1: 0.7992
Test Accuracy: 88.56%
Test Balanced Accuracy: 0.8372
Test MCC: 0.7830


## Classification Report

In [44]:
print(

    classification_report(

        test_tab_results[
            "y_true"
        ],

        test_tab_results[
            "y_pred"
        ],

        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],

        digits=4,

        zero_division=0

    )

)

              precision    recall  f1-score   support

      Benign     0.9306    0.8623    0.8951      3435
  Pathogenic     0.4944    0.7374    0.5919       476
         VUS     0.9091    0.9120    0.9105      5726

    accuracy                         0.8856      9637
   macro avg     0.7780    0.8372    0.7992      9637
weighted avg     0.8963    0.8856    0.8893      9637



# Arm 1 + Arm 2

##  Creating Dataset

In [45]:
class FusionDataset(Dataset):

    def __init__(
        self,
        dataframe,
        tabular_matrix
    ):

        self.df = (
            dataframe
            .reset_index(drop=True)
            .copy()
        )

        self.tabular = torch.tensor(
            tabular_matrix,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            self.df["Label"].values,
            dtype=torch.long
        )


    def __len__(self):

        return len(self.df)


    def __getitem__(self, idx):

        row = self.df.iloc[idx]

        seq = encode_variant_sequence(
            row["seq201"],
            row["ReferenceAlleleVCF"],
            row["AlternateAlleleVCF"],
            window_size=101
        )

        seq = torch.tensor(
            seq,
            dtype=torch.float32
        )

        tab = self.tabular[idx]

        label = self.labels[idx]

        return (
            seq,
            tab,
            label
        )

In [46]:
train_fusion_dataset = FusionDataset(
    train_df,
    X_train_tab
)

val_fusion_dataset = FusionDataset(
    val_df,
    X_val_tab
)

test_fusion_dataset = FusionDataset(
    test_df,
    X_test_tab
)


print(
    "Train:",
    len(train_fusion_dataset)
)

print(
    "Validation:",
    len(val_fusion_dataset)
)

print(
    "Test:",
    len(test_fusion_dataset)
)

Train: 80065
Validation: 10298
Test: 9637


## Dataloaders

In [47]:
FUSION_BATCH_SIZE = (
    512
    if torch.cuda.is_available()
    else 256
)


train_fusion_loader = DataLoader(
    train_fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

val_fusion_loader = DataLoader(
    val_fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)

test_fusion_loader = DataLoader(
    test_fusion_dataset,
    batch_size=FUSION_BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=torch.cuda.is_available()
)


print(
    "Batch size:",
    FUSION_BATCH_SIZE
)

print(
    "Train batches:",
    len(train_fusion_loader)
)

Batch size: 256
Train batches: 313


## Fusion Model

In [59]:
class GatedTwoArmFusionModel(nn.Module):

    def __init__(
        self,
        tabular_input_dim,
        num_classes=3
    ):

        super().__init__()

        # ====================================================
        # ARM 1 — SEQUENCE
        # ====================================================

        self.sequence_encoder = SequenceCNN(
            embedding_dim=64
        )


        # Reduce sequence representation:
        # 64 -> 32
        self.sequence_projection = nn.Sequential(

            nn.Linear(
                64,
                32
            ),

            nn.ReLU(),

            nn.LayerNorm(
                32
            )
        )


        # ====================================================
        # ARM 2 — BIOMARKERS
        # ====================================================

        self.biomarker_encoder = BiomarkerMLP(

            input_dim=tabular_input_dim,

            embedding_dim=32
        )


        self.biomarker_norm = nn.LayerNorm(
            32
        )


        # ====================================================
        # LEARNED ALPHA GATE
        #
        # Input:
        # 32 sequence
        # +
        # 32 biomarker
        # =
        # 64
        #
        # Output:
        # one scalar alpha per variant
        # ====================================================

        self.gate = nn.Linear(
            64,
            1
        )


        # Start by trusting biomarkers more.
        #
        # sigmoid(-1.5) ≈ 0.18
        nn.init.zeros_(
            self.gate.weight
        )

        nn.init.constant_(
            self.gate.bias,
            -1.5
        )


        # ====================================================
        # CLASSIFIER
        # ====================================================

        self.classifier = nn.Sequential(

            nn.Linear(
                32,
                32
            ),

            nn.ReLU(),

            nn.Dropout(
                0.30
            ),

            nn.Linear(
                32,
                num_classes
            )
        )


    def forward(
        self,
        seq,
        tab,
        return_alpha=False
    ):

        # ----------------------------------------------------
        # Sequence representation
        # ----------------------------------------------------

        seq_features = self.sequence_encoder(
            seq
        )

        seq_features = self.sequence_projection(
            seq_features
        )


        # ----------------------------------------------------
        # Biomarker representation
        # ----------------------------------------------------

        tab_features = self.biomarker_encoder(
            tab
        )

        tab_features = self.biomarker_norm(
            tab_features
        )


        # ----------------------------------------------------
        # Compute alpha
        # ----------------------------------------------------

        gate_input = torch.cat(
            [
                seq_features,
                tab_features
            ],
            dim=1
        )


        alpha = torch.sigmoid(
            self.gate(
                gate_input
            )
        )


        # ----------------------------------------------------
        # BIOMARKER-ANCHORED FUSION
        # ----------------------------------------------------

        fused = (
            tab_features
            +
            alpha
            *
            seq_features
        )


        logits = self.classifier(
            fused
        )


        if return_alpha:

            return (
                logits,
                alpha
            )


        return logits

In [60]:
gated_model = GatedTwoArmFusionModel(

    tabular_input_dim=
        X_train_tab.shape[1],

    num_classes=3

).to(device)

## Loading pretrained sequence encoder

In [61]:
sequence_checkpoint = torch.load(
    best_model_path,
    map_location=device
)


sequence_state = {

    key.replace(
        "encoder.",
        "",
        1
    ): value

    for key, value in
    sequence_checkpoint[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}


gated_model.sequence_encoder.load_state_dict(
    sequence_state
)


print(
    "✅ Sequence encoder restored."
)

✅ Sequence encoder restored.


## Loading biomarker encoder

In [62]:
biomarker_checkpoint = torch.load(
    best_tab_path,
    map_location=device
)


biomarker_state = {

    key.replace(
        "encoder.",
        "",
        1
    ): value

    for key, value in
    biomarker_checkpoint[
        "model_state_dict"
    ].items()

    if key.startswith(
        "encoder."
    )
}


gated_model.biomarker_encoder.load_state_dict(
    biomarker_state
)


print(
    "✅ Biomarker encoder restored."
)

✅ Biomarker encoder restored.


## Checking Starting point of Alpha gate

In [63]:
seq_batch, tab_batch, labels = next(
    iter(train_fusion_loader)
)


seq_batch = seq_batch.to(
    device
)

tab_batch = tab_batch.to(
    device
)


with torch.no_grad():

    logits, alpha = gated_model(

        seq_batch,

        tab_batch,

        return_alpha=True
    )


print(
    "Output shape:",
    logits.shape
)

print(
    "Alpha shape:",
    alpha.shape
)

print(
    "Mean alpha:",
    alpha.mean().item()
)

print(
    "Minimum alpha:",
    alpha.min().item()
)

print(
    "Maximum alpha:",
    alpha.max().item()
)

Output shape: torch.Size([256, 3])
Alpha shape: torch.Size([256, 1])
Mean alpha: 0.18242551386356354
Minimum alpha: 0.18242552876472473
Maximum alpha: 0.18242552876472473


## Stage 1: train only gate + projection + classifier

In [65]:
for param in gated_model.sequence_encoder.parameters():
    param.requires_grad = False

for param in gated_model.biomarker_encoder.parameters():
    param.requires_grad = False

### Optimizer

In [66]:
gate_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        gated_model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)

gate_loss_fn = nn.CrossEntropyLoss(
    weight=class_weights
)

### Training

In [76]:
def train_gated_model(
    model,
    train_loader,
    val_loader,
    optimizer,
    loss_fn,
    epochs,
    stage_name,
    checkpoint_path
):

    best_val_f1 = -1.0
    no_improvement = 0
    patience = 3

    for epoch in range(epochs):

        # ============================
        # TRAIN
        # ============================

        model.train()

        running_loss = 0.0

        progress = tqdm(
            train_loader,
            desc=f"{stage_name} {epoch+1}/{epochs}"
        )

        for seq, tab, targets in progress:

            seq = seq.to(device)
            tab = tab.to(device)
            targets = targets.to(device)

            optimizer.zero_grad(
                set_to_none=True
            )

            logits = model(
                seq,
                tab
            )

            loss = loss_fn(
                logits,
                targets
            )

            loss.backward()

            torch.nn.utils.clip_grad_norm_(
                model.parameters(),
                5.0
            )

            optimizer.step()

            running_loss += (
                loss.item()
                * seq.size(0)
            )

            progress.set_postfix(
                loss=f"{loss.item():.4f}"
            )


        train_loss = (
            running_loss
            /
            len(train_fusion_dataset)
        )


        # ============================
        # VALIDATION
        # ============================

        val_results = evaluate_fusion_model(
            model,
            val_loader
        )

        val_f1 = val_results[
            "macro_f1"
        ]


        # ============================
        # ALPHA STATISTICS
        # ============================

        model.eval()

        alpha_values = []

        with torch.no_grad():

            for seq, tab, _ in val_loader:

                seq = seq.to(device)
                tab = tab.to(device)

                _, alpha = model(
                    seq,
                    tab,
                    return_alpha=True
                )

                alpha_values.append(
                    alpha.cpu()
                )

        alpha_values = torch.cat(
            alpha_values,
            dim=0
        )


        mean_alpha = (
            alpha_values.mean().item()
        )

        std_alpha = (
            alpha_values.std().item()
        )


        print(
            f"Epoch {epoch+1:02d} | "
            f"Loss: {train_loss:.4f} | "
            f"Val Acc: {val_results['accuracy']:.4f} | "
            f"Val Balanced Acc: {val_results['balanced_accuracy']:.4f} | "
            f"Val Macro-F1: {val_f1:.4f} | "
            f"Val MCC: {val_results['mcc']:.4f} | "
            f"Alpha Mean: {mean_alpha:.4f} | "
            f"Alpha Std: {std_alpha:.4f}"
        )


        # ============================
        # CHECKPOINT
        # ============================

        if val_f1 > best_val_f1 + 1e-4:

            best_val_f1 = val_f1
            no_improvement = 0

            torch.save(
                {
                    "model_state_dict":
                        model.state_dict(),

                    "epoch":
                        epoch + 1,

                    "val_macro_f1":
                        best_val_f1,

                    "mean_alpha":
                        mean_alpha,

                    "std_alpha":
                        std_alpha
                },
                checkpoint_path
            )

            print("✓ Best model saved.")

        else:

            no_improvement += 1

            print(
                f"No improvement: "
                f"{no_improvement}/{patience}"
            )


        if no_improvement >= patience:

            print(
                "\nEarly stopping triggered."
            )

            break

In [77]:
for param in gated_model.sequence_encoder.parameters():
    param.requires_grad = False

for param in gated_model.biomarker_encoder.parameters():
    param.requires_grad = False


stage1_optimizer = torch.optim.AdamW(
    filter(
        lambda p: p.requires_grad,
        gated_model.parameters()
    ),
    lr=1e-3,
    weight_decay=1e-4
)


train_gated_model(
    model=gated_model,
    train_loader=train_fusion_loader,
    val_loader=val_fusion_loader,
    optimizer=stage1_optimizer,
    loss_fn=gate_loss_fn,
    epochs=3,
    stage_name="Stage 1 Gate Warmup",
    checkpoint_path=
        CHECKPOINT_DIR /
        "gated_stage1_best.pt"
)

Stage 1 Gate Warmup 1/3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.4551 | Val Acc: 0.8713 | Val Balanced Acc: 0.8319 | Val Macro-F1: 0.7767 | Val MCC: 0.7597 | Alpha Mean: 0.5828 | Alpha Std: 0.1020
✓ Best model saved.


Stage 1 Gate Warmup 2/3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 02 | Loss: 0.4495 | Val Acc: 0.8845 | Val Balanced Acc: 0.8344 | Val Macro-F1: 0.7982 | Val MCC: 0.7797 | Alpha Mean: 0.5500 | Alpha Std: 0.1173
✓ Best model saved.


Stage 1 Gate Warmup 3/3:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 03 | Loss: 0.4511 | Val Acc: 0.8617 | Val Balanced Acc: 0.8336 | Val Macro-F1: 0.7658 | Val MCC: 0.7462 | Alpha Mean: 0.6240 | Alpha Std: 0.1111
No improvement: 1/3


### Check Alpha

In [80]:
seq_batch, tab_batch, labels = next(
    iter(val_fusion_loader)
)

seq_batch = seq_batch.to(device)
tab_batch = tab_batch.to(device)

gated_model.eval()

with torch.no_grad():

    logits, alpha = gated_model(
        seq_batch,
        tab_batch,
        return_alpha=True
    )

print("Mean alpha:", alpha.mean().item())
print("Min alpha:", alpha.min().item())
print("Max alpha:", alpha.max().item())
print("Std alpha:", alpha.std().item())

Mean alpha: 0.6808328628540039
Min alpha: 0.41472360491752625
Max alpha: 0.8438806533813477
Std alpha: 0.09222152084112167


In [74]:
torch.save(
    checkpoint,
    CHECKPOINT_DIR / "gated_stage1_best.pt"
)

## Stage 2: Unfreeze Everything

In [69]:
for param in gated_model.parameters():
    param.requires_grad = True

### Optimizer

In [70]:
gate_optimizer = torch.optim.AdamW(
    [
        {
            "params":
                gated_model
                .sequence_encoder
                .parameters(),
            "lr": 5e-5
        },

        {
            "params":
                gated_model
                .biomarker_encoder
                .parameters(),
            "lr": 1e-4
        },

        {
            "params":
                gated_model
                .sequence_projection
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .gate
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .classifier
                .parameters(),
            "lr": 3e-4
        }
    ],

    weight_decay=1e-4
)

### Training

In [78]:
stage1_checkpoint = torch.load(
    CHECKPOINT_DIR /
    "gated_stage1_best.pt",
    map_location=device
)

gated_model.load_state_dict(
    stage1_checkpoint[
        "model_state_dict"
    ]
)

<All keys matched successfully>

In [79]:
for param in gated_model.parameters():
    param.requires_grad = True


stage2_optimizer = torch.optim.AdamW(
    [
        {
            "params":
                gated_model
                .sequence_encoder
                .parameters(),
            "lr": 5e-5
        },

        {
            "params":
                gated_model
                .biomarker_encoder
                .parameters(),
            "lr": 1e-4
        },

        {
            "params":
                gated_model
                .sequence_projection
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .gate
                .parameters(),
            "lr": 3e-4
        },

        {
            "params":
                gated_model
                .classifier
                .parameters(),
            "lr": 3e-4
        }
    ],
    weight_decay=1e-4
)


train_gated_model(
    model=gated_model,
    train_loader=train_fusion_loader,
    val_loader=val_fusion_loader,
    optimizer=stage2_optimizer,
    loss_fn=gate_loss_fn,
    epochs=10,
    stage_name="Stage 2 Fine-Tuning",
    checkpoint_path=
        CHECKPOINT_DIR /
        "gated_stage2_best.pt"
)

Stage 2 Fine-Tuning 1/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 01 | Loss: 0.4451 | Val Acc: 0.8669 | Val Balanced Acc: 0.8362 | Val Macro-F1: 0.7736 | Val MCC: 0.7537 | Alpha Mean: 0.5730 | Alpha Std: 0.1215
✓ Best model saved.


Stage 2 Fine-Tuning 2/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 02 | Loss: 0.4417 | Val Acc: 0.8549 | Val Balanced Acc: 0.8384 | Val Macro-F1: 0.7599 | Val MCC: 0.7377 | Alpha Mean: 0.5514 | Alpha Std: 0.1230
No improvement: 1/3


Stage 2 Fine-Tuning 3/10:   0%|          | 0/313 [00:00<?, ?it/s]

...

Epoch 05 | Loss: 0.4299 | Val Acc: 0.8609 | Val Balanced Acc: 0.8347 | Val Macro-F1: 0.7657 | Val MCC: 0.7452 | Alpha Mean: 0.6337 | Alpha Std: 0.1060
No improvement: 1/3


Stage 2 Fine-Tuning 6/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 06 | Loss: 0.4235 | Val Acc: 0.8618 | Val Balanced Acc: 0.8368 | Val Macro-F1: 0.7671 | Val MCC: 0.7469 | Alpha Mean: 0.6515 | Alpha Std: 0.1017
No improvement: 2/3


Stage 2 Fine-Tuning 7/10:   0%|          | 0/313 [00:00<?, ?it/s]

Epoch 07 | Loss: 0.4260 | Val Acc: 0.8589 | Val Balanced Acc: 0.8388 | Val Macro-F1: 0.7649 | Val MCC: 0.7430 | Alpha Mean: 0.6725 | Alpha Std: 0.0941
No improvement: 3/3

Early stopping triggered.


In [81]:
stage2_checkpoint = torch.load(
    CHECKPOINT_DIR / "gated_stage2_best.pt",
    map_location=device
)

gated_model.load_state_dict(
    stage2_checkpoint["model_state_dict"]
)

print("Best epoch:",
      stage2_checkpoint["epoch"])

print("Best Val Macro-F1:",
      stage2_checkpoint["val_macro_f1"])

print("Mean alpha:",
      stage2_checkpoint["mean_alpha"])

print("Alpha std:",
      stage2_checkpoint["std_alpha"])

Best epoch: 4
Best Val Macro-F1: 0.7915132989945858
Mean alpha: 0.6232147812843323
Alpha std: 0.10979373008012772


## Testing

In [84]:
test_gated_results = evaluate_fusion_model(
    gated_model,
    test_fusion_loader
)

print(
    "Test Accuracy:",
    test_gated_results["accuracy"]
)

print(
    "Test Balanced Accuracy:",
    test_gated_results["balanced_accuracy"]
)

print(
    "Test Macro-F1:",
    test_gated_results["macro_f1"]
)

print(
    "Test MCC:",
    test_gated_results["mcc"]
)

Test Accuracy: 0.8827435924042751
Test Balanced Accuracy: 0.834854527967645
Test Macro-F1: 0.794609912539816
Test MCC: 0.7780371971248816


## Classification Report

In [85]:
print(
    classification_report(
        test_gated_results["y_true"],
        test_gated_results["y_pred"],
        target_names=[
            "Benign",
            "Pathogenic",
            "VUS"
        ],
        digits=4,
        zero_division=0
    )
)

              precision    recall  f1-score   support

      Benign     0.9294    0.8664    0.8968      3435
  Pathogenic     0.4807    0.7332    0.5807       476
         VUS     0.9077    0.9050    0.9063      5726

    accuracy                         0.8827      9637
   macro avg     0.7726    0.8349    0.7946      9637
weighted avg     0.8943    0.8827    0.8869      9637



# New Section